In [31]:
import pandas as pd
import numpy as np

features = pd.read_csv("../data/customer_features.csv")
test = pd.read_csv("../data/customer_clv_test.csv")

test_df = test.merge(features, on="cust_id", how="left")

# number of customers with any missing values
print(test_df.isna().any(axis=1).sum())

# see which features have missing values and how many
print(test_df.isna().sum()[test_df.isna().sum() > 0])

test_df = test_df.fillna(0)

0
Series([], dtype: int64)


In [32]:
import joblib

feature_cols = joblib.load("../models/feature_columns.pkl")
X_test = test_df[feature_cols]

In [33]:
churn_model = joblib.load("../models/churn_cat_model.pkl")
revenue_model = joblib.load("../models/rev_model.pkl")

In [34]:
p_return = churn_model.predict_proba(X_test)[:, 1]

In [35]:
log_rev_pred = revenue_model.predict(X_test)
rev_pred = np.expm1(log_rev_pred)
rev_pred = np.maximum(rev_pred, 0)

threshold = joblib.load("../models/best_threshold.pkl")


final_pred = np.where(
    p_return < threshold,
    0,
    p_return * rev_pred
)

In [36]:
import os
os.makedirs("../submissions", exist_ok=True)

In [37]:
submission = pd.DataFrame({
    "cust_id": test_df["cust_id"],
    "prediction": final_pred
})

submission.to_csv("../submissions/submission_v1.csv", index=False)